# Day 13/365 — Classification Thresholds

## Goal
Logistic Regression and many other classifiers output a **probability or score**.

Today we test what happens when the **same probabilities** are converted into classes using different thresholds: `0.3`, `0.5`, and `0.8`.

> Synthetic educational example only.


## Core idea

If a model predicts:

**Probability of default = 0.48**

then:

- threshold `0.3` → **Default**
- threshold `0.5` → **No Default**
- threshold `0.8` → **No Default**

The probability did not change. Only the decision rule changed.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

probabilities = np.array([0.10, 0.25, 0.35, 0.48, 0.55, 0.67, 0.82, 0.93])
customers = [f"Customer {i}" for i in range(1, 9)]

df = pd.DataFrame({
    "Customer": customers,
    "Default Probability": probabilities
})

df


## 1. Apply different thresholds

We use:

- `1` = Predicted Default
- `0` = Predicted No Default


In [ ]:
def classify(probabilities, threshold):
    return (probabilities >= threshold).astype(int)

thresholds = [0.3, 0.5, 0.8]

for threshold in thresholds:
    df[f"Prediction @ {threshold}"] = classify(probabilities, threshold)

df


### Observation

Focus on probability `0.48`.

The same customer changes class depending on the threshold.

That is the key concept: **probability and final class are not the same thing**.


## 2. Count predicted defaults at each threshold

In [ ]:
positive_predictions = [
    classify(probabilities, t).sum()
    for t in thresholds
]

summary = pd.DataFrame({
    "Threshold": thresholds,
    "Predicted Defaults": positive_predictions
})

summary


In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    [str(t) for t in thresholds],
    positive_predictions
)

plt.xlabel("Classification Threshold")
plt.ylabel("Number of Predicted Defaults")
plt.title("How Threshold Changes Final Predictions")
plt.tight_layout()
plt.show()


## What does this visual prove?

The model probabilities are identical in all three scenarios.

Yet:

- a **lower threshold** produces more positive predictions
- a **higher threshold** produces fewer positive predictions

So the threshold changes the final decision without changing the underlying model score.


## 3. Show each probability against the thresholds

In [ ]:
x = np.arange(len(customers))

plt.figure(figsize=(10, 6))

plt.scatter(
    x,
    probabilities,
    s=90,
    label="Predicted probability"
)

plt.axhline(0.3, linestyle="--", label="Threshold = 0.3")
plt.axhline(0.5, linestyle="--", label="Threshold = 0.5")
plt.axhline(0.8, linestyle="--", label="Threshold = 0.8")

plt.xticks(x, customers, rotation=45)
plt.ylim(0, 1)
plt.ylabel("Predicted Probability of Default")
plt.title("Same Probabilities, Different Decision Thresholds")
plt.legend()
plt.tight_layout()
plt.show()


## 4. One-customer demonstration

In [ ]:
customer_probability = 0.48

for threshold in thresholds:
    prediction = int(customer_probability >= threshold)
    label = "Default" if prediction == 1 else "No Default"

    print(
        f"Probability = {customer_probability}, "
        f"Threshold = {threshold} -> {label}"
    )


# Why threshold selection matters

If missing a true positive is costly, we may lower the threshold to flag more cases.

But that can also increase false positives.

If we raise the threshold, the model becomes more selective, but we may miss more genuine positive cases.

That trade-off leads directly into the next topic: **Confusion Matrix, False Positives, and False Negatives**.


# Final takeaway

> **Same model probabilities + different threshold = different final predictions**

Conceptually:

**Model → Probability → Threshold → Final Class**

The threshold is part of the decision system, not a universal constant.


# Limitations

- Probabilities are synthetic.
- The dataset is intentionally tiny for visual clarity.
- Real thresholds should not be chosen arbitrarily.
- Production threshold selection can depend on costs, risk tolerance, fairness, regulation, and operational constraints.


# References

Google Machine Learning Crash Course — Classification Thresholds  
https://developers.google.com/machine-learning/crash-course/classification/thresholding

Scikit-learn — Tuning the decision threshold for class prediction  
https://scikit-learn.org/stable/modules/classification_threshold.html
